# Hari 006e — HMC/NUTS dari Nol untuk Volatilitas Stokastik Bayesian & Kalibrasi SBC

| Hari | Tanggal | Jenis | Bahasa | Topik |
|:----:|:-------:|:-----:|:------:|-------|
| `006e` | `2026-11-24` | `sim` | `python` | HMC/NUTS dari Nol untuk Volatilitas Stokastik Bayesian & Kalibrasi SBC |

> Bagian dari **100QB** — implementasi mandiri *Hamiltonian Monte Carlo* (HMC) dan *No-U-Turn Sampler* (NUTS) multinomial beradaptasi ganda (*dual averaging* + matriks massa diagonal berjendela) hanya dengan **NumPy + SciPy**, diterapkan pada model **Volatilitas Stokastik (SV)** Bayesian dan diverifikasi ketat melalui **Simulation-Based Calibration (SBC)**.

---

## 1. Setup, Reproduksibilitas (`SeedSequence`) & Kriteria A Priori

Seluruh aliran bilangan acak diturunkan secara deterministik dari satu *seed* induk `20261124` menggunakan `np.random.SeedSequence.spawn` per eksperimen sehingga eksekusi ulang dari awal (*Restart & Run All*) selalu menghasilkan angka yang identik bit-demi-bit. Ambang keberhasilan `c1`–`c10` dikunci *a priori* sebelum perhitungan apa pun dijalankan.


In [1]:
from __future__ import annotations

import json
import math
import multiprocessing as mp
import os
import platform
import sys
import time
import xml.etree.ElementTree as ET
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import scipy
from scipy.signal import lfilter
from scipy.special import betaln, logsumexp, ndtri
from scipy.stats import chi2, rankdata

# Temukan root repositori secara relatif tanpa mencetak path absolut
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "qb100").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from qb100 import plotting
plotting.set_style()

T_NOTEBOOK_START = time.perf_counter()
MASTER_SEED = 20261124
SS_MASTER = np.random.SeedSequence(MASTER_SEED)
(
    SS_FD,
    SS_LF,
    SS_G50,
    SS_SV_DATA,
    SS_SV_NCP,
    SS_SV_CP,
    SS_SBC,
    SS_APP,
    SS_PROF,
) = SS_MASTER.spawn(9)

CRITERIA_APRIORI = {
    "c1_analytic_grad_vs_fd": "maks galat relatif gradien analitik vs selisih hingga terpusat <= 1e-6",
    "c2_leapfrog_reversible_and_order2": "reversibilitas maju-balik <= 1e-9 dan kemiringan log-log galat energi dalam [1.95, 2.05]",
    "c3_nuts_gaussian_50d": "maks |mean_hat - mean| <= 4 SE (ESS) dan galat relatif variansi diagonal <= 0.15",
    "c4_sv_synthetic_convergence": "split-Rhat < 1.01 untuk (mu, phi, sigma_eta), ESS bulk >= 400, divergensi == 0",
    "c5_sv_latent_h_coverage_90": "liputan kredibel 90% jalur laten h_t berada dalam [0.85, 0.95]",
    "c6_sbc_uniformity": "p-value chi-kuadrat > 0.01 untuk ketiga parameter dan ECDF-diff dalam pita simultan 99%",
    "c7_ncp_beats_centered_ess_per_sec": "ESS/detik parameterisasi non-terpusat > terpusat pada seri SV yang sama",
    "c8_clean_reproducible_under_10min": "notebook berjalan bersih <= 600 detik dan dua eksekusi menghasilkan angka identik",
    "c9_unit_asserts_pass": "seluruh sel uji assert lulus tanpa pengecualian",
    "c10_figures_valid": "6 pasang grafik PNG+SVG di outputs/figures/day006e_* ada, tak kosong, dan SVG valid XML",
}

ENV_INFO = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "matplotlib": matplotlib.__version__,
    "arch": platform.machine(),
    "cpu_cores": os.cpu_count() or 1,
}
print("Lingkungan & Seed Induk:", json.dumps({"seed": MASTER_SEED, **ENV_INFO}, indent=2))


Lingkungan & Seed Induk: {
  "seed": 20261124,
  "python": "3.11.18",
  "numpy": "2.4.6",
  "scipy": "1.17.1",
  "matplotlib": "3.11.2",
  "arch": "arm64",
  "cpu_cores": 10
}


---

## 2. Model Volatilitas Stokastik Bayesian, Geometri *Funnel* & Gradien Analitik (`c1`)

### Spesifikasi Model Generatif dan Prior
Log-return harian $y_t$ ($t = 1, \dots, T$) dimodelkan dengan log-variansi laten AR(1) stasioner:

$$y_t = \exp(h_t / 2)\,\varepsilon_t, \quad \varepsilon_t \stackrel{\text{i.i.d.}}{\sim} \mathcal{N}(0, 1), \qquad h_1 \sim \mathcal{N}\!\left(\mu,\, \frac{\sigma_\eta^2}{1 - \phi^2}\right), \qquad h_t = \mu + \phi(h_{t-1} - \mu) + \sigma_\eta \eta_t, \quad \eta_t \stackrel{\text{i.i.d.}}{\sim} \mathcal{N}(0, 1).$$

Prior parameter struktural $(\mu, \phi, \sigma_\eta)$ ditetapkan sebagai berikut:
- $\mu \sim \mathcal{N}(-9{,}0,\, 2{,}0^2)$ (skala log-variansi harian: $\exp(-9/2) \approx 1{,}11\%$ volatilitas harian),
- $\phi = 2\tilde{\phi} - 1 \in (-1, 1)$ dengan $\tilde{\phi} \sim \mathrm{Beta}(20{,}0,\, 1{,}5)$ ($\mathbb{E}[\phi] \approx 0{,}860$, modus $\phi \approx 0{,}949$),
- $\sigma_\eta \sim \mathrm{HalfNormal}(0{,}5)$ pada $(0, \infty)$ dengan kerapatan $p(\sigma_\eta) = \frac{\sqrt{2}}{0{,}5\sqrt{\pi}}\exp\!\left(-\frac{\sigma_\eta^2}{2 \cdot 0{,}5^2}\right)$.

### Mengapa Parameterisasi Non-Terpusat Menghilangkan *Neal's Funnel*?
Pada **parameterisasi terpusat (CP)** dengan keadaan $(h_{1:T}, \mu, \phi, \sigma_\eta)$, kerapatan bersyarat prior memiliki suku $-\frac{1}{2\sigma_\eta^2}\sum_{t=2}^T ((h_t-\mu) - \phi(h_{t-1}-\mu))^2$. Ketika $\sigma_\eta$ kecil, kelengkungan Hessian sepanjang arah $h_t$ meledak sebanding dengan $\sigma_\eta^{-2}$ (*leher corong / Neal's funnel*), sedangkan untuk $\sigma_\eta$ besar kelengkungannya landai (*mulut corong*). Satu ukuran langkah $\varepsilon$ tunggal tidak mampu menjelajahi kedua rezim sekaligus tanpa mengalami divergensi atau kemacetan rantai.

Pada **parameterisasi non-terpusat (NCP)** berbasis inovasi terstandar, kita menuliskan $h_t = \mu + \sigma_\eta z_t$ dengan proses AR(1) $z_t$ yang dibangun lewat operator resolven kausal $R_\phi$ atas inovasi baku $\eta_{1:T} \sim \mathcal{N}(0, I_T)$:

$$z_1 = \frac{\eta_1}{\sqrt{1 - \phi^2}}, \qquad z_t = \phi z_{t-1} + \eta_t \quad (t = 2, \dots, T), \qquad h_t = \mu + \sigma_\eta z_t.$$

Prior atas koordinat laten $\eta_{1:T}$ kini menjadi Gaussian isotropik $\mathcal{N}(0, I_T)$ yang **bebas dari $(\mu, \phi, \sigma_\eta)$**. Untuk ruang tanpa batas $\theta = (\mu, u_\phi, u_\sigma) \in \mathbb{R}^3$ dengan $u_\phi = \mathrm{logit}(\tilde{\phi})$ dan $u_\sigma = \ln\sigma_\eta$:
- Log-prior + log-Jacobian pada $u_\phi$: $\ln p(u_\phi) = -\ln\mathrm{B}(\alpha_\phi, \beta_\phi) + \alpha_\phi \ln\tilde{\phi} + \beta_\phi\ln(1 - \tilde{\phi})$, dengan $\frac{\mathrm{d}\phi}{\mathrm{d}u_\phi} = 2\tilde{\phi}(1 - \tilde{\phi}) = \frac{1 - \phi^2}{2}$.
- Log-prior + log-Jacobian pada $u_\sigma$: $\ln p(u_\sigma) = \frac{1}{2}\ln\frac{2}{\pi s_\sigma^2} - \frac{\sigma_\eta^2}{2 s_\sigma^2} + u_\sigma$.
- **Backpropagation manual lewat resolven adjoin $R_\phi^\top$**: dari gradien log-likelihood $g_t = \frac{\partial \ell}{\partial h_t} = \frac{1}{2}(y_t^2 e^{-h_t} - 1)$ dan $\bar{z}_t = \sigma_\eta g_t$, rekursi mundur $\lambda_T = \bar{z}_T$, $\lambda_t = \bar{z}_t + \phi\lambda_{t+1}$ ($t = T-1, \dots, 1$) memberikan $\frac{\partial \ell}{\partial \eta_t} = \lambda_t$ ($t \ge 2$), $\frac{\partial \ell}{\partial \eta_1} = \frac{\lambda_1}{\sqrt{1-\phi^2}}$, dan $\frac{\partial \ell}{\partial u_\phi} = \frac{1-\phi^2}{2}\sum_{t=2}^T \lambda_t z_{t-1} + \frac{1}{2}\phi\lambda_1 z_1$ (faktor $(1-\phi^2)$ saling meniadakan secara eksak di $t=1$).


In [2]:
MU0_PRIOR, S_MU_PRIOR = -9.0, 2.0
ALPHA_PHI_PRIOR, BETA_PHI_PRIOR = 20.0, 1.5
S_SIG_PRIOR = 0.5

LOG_2PI = math.log(2.0 * math.pi)
LOG_MU_CONST = -0.5 * math.log(2.0 * math.pi * S_MU_PRIOR * S_MU_PRIOR)
LOG_BETA_CONST = float(betaln(ALPHA_PHI_PRIOR, BETA_PHI_PRIOR))
LOG_HN_CONST = 0.5 * math.log(2.0 / (math.pi * S_SIG_PRIOR * S_SIG_PRIOR))
B_RESOLVENT = np.array([1.0], dtype=float)


def unpack_sv_theta(theta3: np.ndarray) -> tuple[float, ...]:
    """Transformasi stabil (mu, u_phi, u_sig) -> (mu, phi, sigma_eta) bebas pembatalan numerik."""
    mu, u_phi, u_sig = float(theta3[0]), float(theta3[1]), float(theta3[2])
    if not (math.isfinite(mu) and math.isfinite(u_phi) and math.isfinite(u_sig)):
        return mu, u_phi, u_sig, 0.0, 0.0, 0.5, 0.5, 0.0, 0.0, 0.0, 1.0
    au = abs(u_phi)
    soft = math.log1p(math.exp(-au))
    log_pt = -soft - (0.0 if u_phi >= 0.0 else -u_phi)
    log_1mpt = -soft - (u_phi if u_phi >= 0.0 else 0.0)
    pt = math.exp(log_pt)
    ompt = math.exp(log_1mpt)
    phi = pt - ompt  # identik dengan 2*pt - 1 tanpa kehilangan presisi dekat +-1
    s1mp2 = 2.0 * math.exp(0.5 * (log_pt + log_1mpt))  # sqrt(1 - phi^2)
    omp2 = s1mp2 * s1mp2
    sig = math.exp(min(u_sig, 40.0))
    return mu, u_phi, u_sig, log_pt, log_1mpt, pt, ompt, phi, s1mp2, omp2, sig


def reconstruct_h_ncp(q: np.ndarray) -> np.ndarray:
    """Bangun jalur log-variansi h_{1:T} dari vektor keadaan NCP q = [mu, u_phi, u_sig, eta_{1:T}]."""
    mu, _, _, _, _, _, _, phi, s1mp2, _, sig = unpack_sv_theta(q[:3])
    eta_sc = q[3:].copy()
    eta_sc[0] /= s1mp2
    z = lfilter(B_RESOLVENT, np.array([1.0, -phi], dtype=float), eta_sc)
    return mu + sig * z


def sv_ncp_neg_log_post_and_grad(q: np.ndarray, log_y2: np.ndarray) -> tuple[float, np.ndarray]:
    """Energi potensial U(q) = -log p(q | y) dan gradien analitik manual untuk SV Non-Terpusat."""
    with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
        T = log_y2.shape[0]
        mu, u_phi, u_sig, log_pt, log_1mpt, pt, ompt, phi, s1mp2, omp2, sig = unpack_sv_theta(q[:3])
        if u_sig > 15.0 or u_sig < -18.0 or s1mp2 < 1e-12:
            return np.inf, np.zeros_like(q)
        eta = q[3:]
        a_ar = np.array([1.0, -phi], dtype=float)
        eta_sc = eta.copy()
        eta_sc[0] = eta[0] / s1mp2
        z = lfilter(B_RESOLVENT, a_ar, eta_sc)
        h = mu + sig * z
        diff = log_y2 - h
        if float(np.max(diff)) > 500.0:
            return np.inf, np.zeros_like(q)
        exp_diff = np.exp(diff)

        ll = -0.5 * T * LOG_2PI - 0.5 * float(np.sum(h + exp_diff))
        lp_mu = LOG_MU_CONST - 0.5 * ((mu - MU0_PRIOR) / S_MU_PRIOR) ** 2
        lp_phi = -LOG_BETA_CONST + ALPHA_PHI_PRIOR * log_pt + BETA_PHI_PRIOR * log_1mpt
        lp_sig = LOG_HN_CONST - 0.5 * (sig / S_SIG_PRIOR) ** 2 + u_sig
        lp_eta = -0.5 * T * LOG_2PI - 0.5 * float(np.dot(eta, eta))
        log_post = ll + lp_mu + lp_phi + lp_sig + lp_eta
        if not math.isfinite(log_post):
            return np.inf, np.zeros_like(q)

        # Backprop manual melalui operator adjoin resolven AR(1) (penjumlahan balik)
        g_h = 0.5 * (exp_diff - 1.0)
        lam = lfilter(B_RESOLVENT, a_ar, (sig * g_h)[::-1])[::-1]

        grad_mu = float(np.sum(g_h)) - (mu - MU0_PRIOR) / (S_MU_PRIOR * S_MU_PRIOR)
        grad_uphi = (
            0.5 * omp2 * float(np.dot(lam[1:], z[:-1]))
            + 0.5 * phi * lam[0] * z[0]
            + ALPHA_PHI_PRIOR * ompt
            - BETA_PHI_PRIOR * pt
        )
        grad_usig = sig * float(np.dot(g_h, z)) + 1.0 - (sig / S_SIG_PRIOR) ** 2

        grad = np.empty_like(q)
        grad[0] = -grad_mu
        grad[1] = -grad_uphi
        grad[2] = -grad_usig
        grad[3] = -(lam[0] / s1mp2 - eta[0])
        grad[4:] = -(lam[1:] - eta[1:])
        if not np.all(np.isfinite(grad)):
            return np.inf, np.zeros_like(q)
        return -float(log_post), grad


def sv_ncp_grad_python_loop(q: np.ndarray, log_y2: np.ndarray) -> tuple[float, np.ndarray]:
    """Versi loop Python eksplisit untuk verifikasi silang & pembanding efisiensi Bagian 9."""
    T = log_y2.shape[0]
    mu, u_phi, u_sig, log_pt, log_1mpt, pt, ompt, phi, s1mp2, omp2, sig = unpack_sv_theta(q[:3])
    eta = q[3:]
    z = np.empty(T, dtype=float)
    z[0] = eta[0] / s1mp2
    for t in range(1, T):
        z[t] = phi * z[t - 1] + eta[t]
    h = mu + sig * z
    exp_diff = np.exp(log_y2 - h)
    ll = -0.5 * T * LOG_2PI - 0.5 * float(np.sum(h + exp_diff))
    lp = (
        LOG_MU_CONST - 0.5 * ((mu - MU0_PRIOR) / S_MU_PRIOR) ** 2
        - LOG_BETA_CONST + ALPHA_PHI_PRIOR * log_pt + BETA_PHI_PRIOR * log_1mpt
        + LOG_HN_CONST - 0.5 * (sig / S_SIG_PRIOR) ** 2 + u_sig
        - 0.5 * T * LOG_2PI - 0.5 * float(np.dot(eta, eta))
    )
    g_h = 0.5 * (exp_diff - 1.0)
    lam = np.empty(T, dtype=float)
    lam[T - 1] = sig * g_h[T - 1]
    for t in range(T - 2, -1, -1):
        lam[t] = sig * g_h[t] + phi * lam[t + 1]
    grad = np.empty_like(q)
    grad[0] = -(float(np.sum(g_h)) - (mu - MU0_PRIOR) / (S_MU_PRIOR * S_MU_PRIOR))
    grad[1] = -(0.5 * omp2 * float(np.dot(lam[1:], z[:-1])) + 0.5 * phi * lam[0] * z[0] + ALPHA_PHI_PRIOR * ompt - BETA_PHI_PRIOR * pt)
    grad[2] = -(sig * float(np.dot(g_h, z)) + 1.0 - (sig / S_SIG_PRIOR) ** 2)
    grad[3] = -(lam[0] / s1mp2 - eta[0])
    grad[4:] = -(lam[1:] - eta[1:])
    return -float(ll + lp), grad


def sv_cp_neg_log_post_and_grad(q: np.ndarray, log_y2: np.ndarray) -> tuple[float, np.ndarray]:
    """Energi potensial U(q) dan gradien analitik untuk SV Terpusat q = [mu, u_phi, u_sig, h_{1:T}]."""
    with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
        T = log_y2.shape[0]
        mu, u_phi, u_sig, log_pt, log_1mpt, pt, ompt, phi, s1mp2, omp2, sig = unpack_sv_theta(q[:3])
        if u_sig > 15.0 or u_sig < -18.0 or s1mp2 < 1e-12:
            return np.inf, np.zeros_like(q)
        h = q[3:]
        diff = log_y2 - h
        if float(np.max(diff)) > 500.0:
            return np.inf, np.zeros_like(q)
        exp_diff = np.exp(diff)
        inv_sig2 = 1.0 / (sig * sig)

        hm = h - mu
        res = hm[1:] - phi * hm[:-1]
        quad = omp2 * hm[0] * hm[0] + float(np.dot(res, res))

        ll = -0.5 * T * LOG_2PI - 0.5 * float(np.sum(h + exp_diff))
        lp_h = -0.5 * T * LOG_2PI - T * u_sig + 0.5 * math.log(omp2) - 0.5 * inv_sig2 * quad
        lp_mu = LOG_MU_CONST - 0.5 * ((mu - MU0_PRIOR) / S_MU_PRIOR) ** 2
        lp_phi = -LOG_BETA_CONST + ALPHA_PHI_PRIOR * log_pt + BETA_PHI_PRIOR * log_1mpt
        lp_sig = LOG_HN_CONST - 0.5 * (sig / S_SIG_PRIOR) ** 2 + u_sig
        log_post = ll + lp_h + lp_mu + lp_phi + lp_sig
        if not math.isfinite(log_post):
            return np.inf, np.zeros_like(q)

        g_h = 0.5 * (exp_diff - 1.0)
        gp_h = np.zeros_like(h)
        gp_h[0] = -inv_sig2 * omp2 * hm[0] + inv_sig2 * phi * res[0]
        gp_h[1:] -= inv_sig2 * res
        gp_h[1:-1] += inv_sig2 * phi * res[1:]

        grad_mu = inv_sig2 * (omp2 * hm[0] + (1.0 - phi) * float(np.sum(res))) - (mu - MU0_PRIOR) / (S_MU_PRIOR * S_MU_PRIOR)
        grad_uphi = (
            -0.5 * phi
            + 0.5 * omp2 * inv_sig2 * (phi * hm[0] * hm[0] + float(np.dot(res, hm[:-1])))
            + ALPHA_PHI_PRIOR * ompt
            - BETA_PHI_PRIOR * pt
        )
        grad_usig = -T + inv_sig2 * quad + 1.0 - (sig / S_SIG_PRIOR) ** 2

        grad = np.empty_like(q)
        grad[0] = -grad_mu
        grad[1] = -grad_uphi
        grad[2] = -grad_usig
        grad[3:] = -(g_h + gp_h)
        if not np.all(np.isfinite(grad)):
            return np.inf, np.zeros_like(q)
        return -float(log_post), grad


In [3]:
# Sel Uji Assert Bagian 2 (c1): Verifikasi Gradien Analitik vs Selisih Hingga Terpusat
def central_fd_grad(fn, q: np.ndarray, eps_fd: float = 1e-6) -> np.ndarray:
    g_num = np.empty_like(q)
    for k in range(q.size):
        qp = q.copy()
        qm = q.copy()
        qp[k] += eps_fd
        qm[k] -= eps_fd
        g_num[k] = (fn(qp)[0] - fn(qm)[0]) / (2.0 * eps_fd)
    return g_num


rng_fd = np.random.default_rng(SS_FD)
T_FD = 60
y_fd = np.exp(0.5 * (-9.0 + 0.3 * rng_fd.normal(size=T_FD))) * rng_fd.normal(size=T_FD)
ly2_fd = np.log(y_fd * y_fd + 1e-300)

fd_rel_errors_ncp = []
fd_rel_errors_cp = []
for trial, (mu_t, uphi_t, usig_t) in enumerate([(-9.1, 2.7, math.log(0.25)), (-8.2, 4.3, math.log(0.07)), (-10.3, -0.4, math.log(0.60))]):
    q_ncp_t = np.concatenate([np.array([mu_t, uphi_t, usig_t]), rng_fd.normal(size=T_FD)])
    u_vec, g_vec = sv_ncp_neg_log_post_and_grad(q_ncp_t, ly2_fd)
    u_loop, g_loop = sv_ncp_grad_python_loop(q_ncp_t, ly2_fd)
    g_fd_ncp = central_fd_grad(lambda x: sv_ncp_neg_log_post_and_grad(x, ly2_fd), q_ncp_t)
    rel_ncp = float(np.max(np.abs(g_vec - g_fd_ncp)) / max(1.0, float(np.max(np.abs(g_fd_ncp)))))
    fd_rel_errors_ncp.append(rel_ncp)
    assert abs(u_vec - u_loop) <= 1e-12 and np.max(np.abs(g_vec - g_loop)) <= 1e-12

    q_cp_t = np.concatenate([np.array([mu_t, uphi_t, usig_t]), mu_t + 0.35 * rng_fd.normal(size=T_FD)])
    _, g_cp = sv_cp_neg_log_post_and_grad(q_cp_t, ly2_fd)
    g_fd_cp = central_fd_grad(lambda x: sv_cp_neg_log_post_and_grad(x, ly2_fd), q_cp_t)
    rel_cp = float(np.max(np.abs(g_cp - g_fd_cp)) / max(1.0, float(np.max(np.abs(g_fd_cp)))))
    fd_rel_errors_cp.append(rel_cp)

C1_MAX_REL_ERR = max(max(fd_rel_errors_ncp), max(fd_rel_errors_cp))
print(f"[c1] Maks galat relatif FD (NCP): {max(fd_rel_errors_ncp):.3e} | (CP): {max(fd_rel_errors_cp):.3e} | gabungan: {C1_MAX_REL_ERR:.3e}")
assert C1_MAX_REL_ERR <= 1e-6, f"Pelanggaran c1: {C1_MAX_REL_ERR}"


[c1] Maks galat relatif FD (NCP): 9.577e-09 | (CP): 8.799e-10 | gabungan: 9.577e-09


---

## 3. Integrator Leapfrog Tervektorisasi, Dual Averaging & Adaptasi Massa Berjendela (`c2`)

Untuk Hamiltonian $H(q, p) = U(q) + \frac{1}{2} p^\top M^{-1} p$ dengan matriks massa diagonal $M = \mathrm{diag}(m)$ (`inv_m` $= m^{-1}$), satu langkah **Velocity Verlet / Leapfrog** berukuran $\varepsilon$ memperbarui $(q, p)$ secara simetris waktu:

$$p_{t + \varepsilon/2} = p_t - \frac{\varepsilon}{2}\nabla U(q_t), \qquad q_{t + \varepsilon} = q_t + \varepsilon M^{-1} p_{t + \varepsilon/2}, \qquad p_{t + \varepsilon} = p_{t + \varepsilon/2} - \frac{\varepsilon}{2}\nabla U(q_{t + \varepsilon}).$$

Komposisi geseran setengah-langkah momentum dan geseran penuh posisi masing-masing memiliki determinan Jacobian ruang fasa bernilai $1$, sehingga pemetaan *leapfrog* bersifat **simplektik** ($\det \frac{\partial(q_L, p_L)}{\partial(q_0, p_0)} = 1$), **reversibel** di bawah pembalikan momentum $p \mapsto -p$, dan memiliki galat Hamiltonian global orde kedua $|\Delta H(\varepsilon)| = O(\varepsilon^2)$ untuk waktu integrasi tetap $\tau = L\varepsilon$.

Selama fase *warmup*, ukuran langkah $\varepsilon$ diadaptasi dengan skema **dual averaging** Nesterov–Hoffman–Gelman ($\gamma = 0{,}05, t_0 = 10, \kappa = 0{,}75$) sementara variansi marginal `inv_m` dipelajari lewat **jendela lambat yang mengganda** (*Stan windowed adaptation*: *initial fast buffer* $\to$ jendela kovarians diagonal ter-regularisasi $\to$ *terminal fast buffer*).


In [4]:
def leapfrog_step(q: np.ndarray, p: np.ndarray, grad: np.ndarray, step: float, inv_m: np.ndarray, fn):
    """Satu langkah integrator simplektik leapfrog tervektorisasi."""
    with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
        p_half = p - 0.5 * step * grad
        q_new = q + step * (inv_m * p_half)
        u_new, grad_new = fn(q_new)
        if not math.isfinite(u_new):
            return q_new, p_half, np.inf, grad_new, np.inf
        p_new = p_half - 0.5 * step * grad_new
        ke_new = 0.5 * float(np.dot(p_new * p_new, inv_m))
        h_new = u_new + ke_new if math.isfinite(ke_new) else np.inf
        return q_new, p_new, u_new, grad_new, h_new


def hmc_integrate(q0: np.ndarray, p0: np.ndarray, eps: float, n_steps: int, inv_m: np.ndarray, fn):
    """Integrasi lintasan L langkah leapfrog untuk HMC statis & verifikasi reversibilitas."""
    q = q0.copy()
    p = p0.copy()
    u, grad = fn(q)
    h = u + 0.5 * float(np.dot(p * p, inv_m))
    for _ in range(n_steps):
        q, p, u, grad, h = leapfrog_step(q, p, grad, eps, inv_m, fn)
    return q, p, u, grad, h


def find_reasonable_epsilon(q: np.ndarray, u: float, grad: np.ndarray, inv_m: np.ndarray, fn, rng, eps_hint: float = 0.1) -> float:
    """Heuristik Hoffman-Gelman (2014) mencari ukuran langkah awal dengan peluang terima sekitar 0.8."""
    eps = float(np.clip(eps_hint, 1e-4, 1.0))
    p = rng.normal(size=q.shape[0]) / np.sqrt(inv_m)
    h0 = u + 0.5 * float(np.dot(p * p, inv_m))
    _, _, _, _, hn = leapfrog_step(q, p, grad, eps, inv_m, fn)
    log_a = -(hn - h0) if math.isfinite(hn) else -np.inf
    target = math.log(0.8)
    direction = 1 if log_a > target else -1
    for _ in range(20):
        if (direction == 1 and log_a <= target) or (direction == -1 and log_a >= target):
            break
        eps *= 2.0 if direction == 1 else 0.5
        if eps < 1e-5 or eps > 5.0:
            break
        _, _, _, _, hn = leapfrog_step(q, p, grad, eps, inv_m, fn)
        log_a = -(hn - h0) if math.isfinite(hn) else -np.inf
    return float(np.clip(eps, 1e-4, 2.0))


def build_stan_warmup_windows(n_warmup: int, adapt_mass: bool) -> tuple[int, int, set[int]]:
    """Bentuk jadwal warmup 3-tahap ala Stan: init buffer, jendela massa mengganda, terminal buffer."""
    init_buf = max(20, int(0.15 * n_warmup))
    term_buf = max(50, int(0.20 * n_warmup))
    slow_len = max(0, n_warmup - init_buf - term_buf)
    win_ends: set[int] = set()
    if adapt_mass and slow_len >= 25:
        cur = init_buf
        w = 25
        while cur + w < init_buf + slow_len:
            if cur + 2 * w > init_buf + slow_len:
                win_ends.add(init_buf + slow_len - 1)
                break
            win_ends.add(cur + w - 1)
            cur += w
            w *= 2
        else:
            win_ends.add(init_buf + slow_len - 1)
    return init_buf, slow_len, win_ends


In [5]:
# Sel Uji Assert Bagian 3 (c2): Reversibilitas, Simplektisitas, dan Penskalaan Energi O(eps^2)
rng_lf = np.random.default_rng(SS_LF)
q_lf0 = np.concatenate([np.array([-9.0, 2.9, math.log(0.25)]), 0.5 * rng_lf.normal(size=T_FD)])
inv_m_lf = rng_lf.uniform(0.5, 1.8, size=q_lf0.shape[0])
p_lf0 = rng_lf.normal(size=q_lf0.shape[0]) / np.sqrt(inv_m_lf)
fn_lf = lambda x: sv_ncp_neg_log_post_and_grad(x, ly2_fd)

# 1. Uji reversibilitas maju L=25 langkah -> balik momentum -> maju L=25 langkah -> balik momentum
q_fwd, p_fwd, _, _, h_fwd = hmc_integrate(q_lf0, p_lf0, eps=0.015, n_steps=25, inv_m=inv_m_lf, fn=fn_lf)
q_rev, p_rev, _, _, _ = hmc_integrate(q_fwd, -p_fwd, eps=0.015, n_steps=25, inv_m=inv_m_lf, fn=fn_lf)
p_rev = -p_rev
C2_REV_ERR = float(max(np.max(np.abs(q_rev - q_lf0)), np.max(np.abs(p_rev - p_lf0))))

# 2. Uji pelestarian volume simplektik pada sistem 1-derajat-kebebasan non-Gaussian (potensial kuartik + SV 1D)
def anharmonic_1d(x: np.ndarray) -> tuple[float, np.ndarray]:
    v = float(x[0])
    return 0.5 * v * v + 0.25 * (v ** 4) + 0.3 * math.sin(v), np.array([v + v ** 3 + 0.3 * math.cos(v)])

q1d, p1d, m1d = np.array([0.7]), np.array([-0.4]), np.array([1.3])
jac_2x2 = np.zeros((2, 2))
h_j = 1e-6
for col, (dq, dp) in enumerate([(h_j, 0.0), (0.0, h_j)]):
    qp1, pp1, _, _, _ = hmc_integrate(q1d + dq, p1d + dp, 0.05, 12, m1d, anharmonic_1d)
    qm1, pm1, _, _, _ = hmc_integrate(q1d - dq, p1d - dp, 0.05, 12, m1d, anharmonic_1d)
    jac_2x2[0, col] = (qp1[0] - qm1[0]) / (2.0 * h_j)
    jac_2x2[1, col] = (pp1[0] - pm1[0]) / (2.0 * h_j)
C2_SYMPLECTIC_DET_ERR = abs(float(np.linalg.det(jac_2x2)) - 1.0)

# 3. Uji orde galat konservasi energi |H(q_L, p_L) - H(q_0, p_0)| ~ O(eps^2) pada tau = L*eps = 0.32 tetap
u_init, _ = fn_lf(q_lf0)
h_init = u_init + 0.5 * float(np.dot(p_lf0 * p_lf0, inv_m_lf))
eps_grid = np.array([0.04, 0.02, 0.01, 0.005, 0.0025])
dH_grid = []
for ep in eps_grid:
    n_st = int(round(0.32 / ep))
    _, _, _, _, h_end = hmc_integrate(q_lf0, p_lf0, float(ep), n_st, inv_m_lf, fn_lf)
    dH_grid.append(abs(h_end - h_init))
C2_ENERGY_SLOPE = float(np.polyfit(np.log(eps_grid), np.log(np.array(dH_grid)), 1)[0])

print(
    f"[c2] Reversibilitas maks |dq, dp| = {C2_REV_ERR:.3e} | "
    f"|det(J) - 1| = {C2_SYMPLECTIC_DET_ERR:.3e} | kemiringan log|dH|/log(eps) = {C2_ENERGY_SLOPE:.4f}"
)
assert C2_REV_ERR <= 1e-9, f"Pelanggaran reversibilitas c2: {C2_REV_ERR}"
assert C2_SYMPLECTIC_DET_ERR <= 1e-8, f"Pelanggaran simplektik: {C2_SYMPLECTIC_DET_ERR}"
assert 1.95 <= C2_ENERGY_SLOPE <= 2.05, f"Pelanggaran orde energi c2: {C2_ENERGY_SLOPE}"


[c2] Reversibilitas maks |dq, dp| = 4.441e-16 | |det(J) - 1| = 1.594e-10 | kemiringan log|dH|/log(eps) = 1.9985


---

## 4. Multinomial NUTS dengan Kriteria U-Turn Generalisasi & Uji Gaussian 50-D (`c3`)

Berbeda dari versi asli Hoffman–Gelman (2014) berbasis variabel iris (*slice variable*), implementasi ini menggunakan **Multinomial NUTS modern** (Betancourt, 2017):
1. **Bobot lintasan Boltzmann**: setiap titik lintasan $z_k = (q_k, p_k)$ diberi bobot tak-ternormalisasi $\log w_k = H(q_0, p_0) - H(z_k)$. Di dalam pembangunan sub-pohon biner (`build_nuts_tree`), kandidat sampel dipilih secara multinomial tak-bias dengan peluang $\frac{w_{\text{baru}}}{w_{\text{lama}} + w_{\text{baru}}}$.
2. **Transisi progresif terbias (*biased progressive sampling*)**: pada penggabungan tingkat luar saat kedalaman pohon digandakan dari $2^d$ ke $2^{d+1}$, apabila sub-pohon ekstensi valid (bebas divergensi dan bebas U-turn internal), sampel sub-pohon ekstensi diterima menggantikan sampel pohon lama dengan peluang Barker/Metropolis $\min\!\bigl(1, \frac{w_{\text{ekstensi}}}{w_{\text{lama}}}\bigr)$ yang memperbesar jarak lompatan efektif sambil menjaga *detailed balance* eksak.
3. **Kriteria U-turn generalisasi dalam metrik $M$**: menggunakan momentum terintegrasi sepanjang sub-lintasan $\rho = \sum_{s} p_s$, penghentian terjadi segera ketika hasil kali dalam dengan kecepatan ujung $v_\pm = M^{-1} p_\pm$ tidak lagi positif ($\rho^\top M^{-1} p_- \le 0$ atau $\rho^\top M^{-1} p_+ \le 0$, beserta pemeriksaan sub-interval antarsambungan sub-pohon yang simetris waktu), dengan batas kedalaman pohon `max_depth = 10`.


In [6]:
def check_merged_no_uturn(
    rho_l: np.ndarray, p_lm: np.ndarray, p_lp: np.ndarray,
    rho_r: np.ndarray, p_rm: np.ndarray, p_rp: np.ndarray,
    inv_m: np.ndarray,
) -> bool:
    """Kriteria U-turn generalisasi (Betancourt 2017) yang simetris waktu saat menggabungkan dua sub-pohon."""
    rho = rho_l + rho_r
    if float(np.dot(rho, inv_m * p_lm)) <= 0.0 or float(np.dot(rho, inv_m * p_rp)) <= 0.0:
        return False
    r_lr = rho_l + p_rm
    if float(np.dot(r_lr, inv_m * p_lm)) <= 0.0 or float(np.dot(r_lr, inv_m * p_rm)) <= 0.0:
        return False
    r_rl = p_lp + rho_r
    if float(np.dot(r_rl, inv_m * p_lp)) <= 0.0 or float(np.dot(r_rl, inv_m * p_rp)) <= 0.0:
        return False
    return True


def build_nuts_tree(
    q: np.ndarray, p: np.ndarray, grad: np.ndarray,
    direction: int, depth: int, eps: float, inv_m: np.ndarray,
    h0: float, fn, rng, delta_max: float = 1000.0,
):
    """Rekursi pohon biner Multinomial NUTS hingga kedalaman tertentu."""
    if depth == 0:
        qn, pn, un, gn, hn = leapfrog_step(q, p, grad, direction * eps, inv_m, fn)
        dH = hn - h0
        div = (not math.isfinite(hn)) or (dH > delta_max)
        log_w = -dH if not div else -np.inf
        alpha = min(1.0, math.exp(min(0.0, -dH))) if math.isfinite(hn) else 0.0
        return qn, pn, gn, qn, pn, gn, qn, un, gn, hn, log_w, not div, div, alpha, 1, pn.copy()

    res1 = build_nuts_tree(q, p, grad, direction, depth - 1, eps, inv_m, h0, fn, rng, delta_max)
    qm1, pm1, gm1, qp1, pp1, gp1, q1, u1, g1, h1, lw1, val1, div1, a1, n1, rho1 = res1
    if not val1:
        return res1

    if direction == -1:
        res2 = build_nuts_tree(qm1, pm1, gm1, -1, depth - 1, eps, inv_m, h0, fn, rng, delta_max)
        qm2, pm2, gm2, qp2, pp2, gp2, q2, u2, g2, h2, lw2, val2, div2, a2, n2, rho2 = res2
        qm, pm, gm, qp, pp, gp = qm2, pm2, gm2, qp1, pp1, gp1
        ut_ok = val2 and check_merged_no_uturn(rho2, pm2, pp2, rho1, pm1, pp1, inv_m)
    else:
        res2 = build_nuts_tree(qp1, pp1, gp1, 1, depth - 1, eps, inv_m, h0, fn, rng, delta_max)
        qm2, pm2, gm2, qp2, pp2, gp2, q2, u2, g2, h2, lw2, val2, div2, a2, n2, rho2 = res2
        qm, pm, gm, qp, pp, gp = qm1, pm1, gm1, qp2, pp2, gp2
        ut_ok = val2 and check_merged_no_uturn(rho1, pm1, pp1, rho2, pm2, pp2, inv_m)

    lw_comb = float(np.logaddexp(lw1, lw2))
    q_prop, u_prop, g_prop, h_prop = q1, u1, g1, h1
    if math.isfinite(lw2) and math.isfinite(lw_comb):
        if rng.random() < math.exp(lw2 - lw_comb):
            q_prop, u_prop, g_prop, h_prop = q2, u2, g2, h2
    return (
        qm, pm, gm, qp, pp, gp, q_prop, u_prop, g_prop, h_prop,
        lw_comb, ut_ok, div1 or div2, a1 + a2, n1 + n2, rho1 + rho2,
    )


def nuts_step(
    q: np.ndarray, u: float, grad: np.ndarray,
    eps: float, inv_m: np.ndarray, sqrt_m: np.ndarray,
    fn, rng, max_depth: int = 10,
):
    """Satu transisi Markov Multinomial NUTS dengan pencabangan progresif terbias."""
    p0 = rng.normal(size=q.shape[0]) * sqrt_m
    h0 = u + 0.5 * float(np.dot(p0 * p0, inv_m))
    qm = qp = q
    pm = pp = p0
    gm = gp = grad
    q_curr, u_curr, g_curr, h_curr = q, u, grad, h0
    lw_old = 0.0
    rho = p0.copy()
    div_any = False
    alpha_sum = 0.0
    n_alpha = 0
    depth = 0

    while depth < max_depth:
        direction = -1 if rng.random() < 0.5 else 1
        if direction == -1:
            res = build_nuts_tree(qm, pm, gm, -1, depth, eps, inv_m, h0, fn, rng)
            qm_e, pm_e, gm_e, _, pp_e, _, q_e, u_e, g_e, h_e, lw_e, val_e, div_e, a_e, n_e, rho_e = res
            ut_ok = val_e and check_merged_no_uturn(rho_e, pm_e, pp_e, rho, pm, pp, inv_m)
            qm, pm, gm = qm_e, pm_e, gm_e
        else:
            res = build_nuts_tree(qp, pp, gp, 1, depth, eps, inv_m, h0, fn, rng)
            _, pm_e, _, qp_e, pp_e, gp_e, q_e, u_e, g_e, h_e, lw_e, val_e, div_e, a_e, n_e, rho_e = res
            ut_ok = val_e and check_merged_no_uturn(rho, pm, pp, rho_e, pm_e, pp_e, inv_m)
            qp, pp, gp = qp_e, pp_e, gp_e

        div_any = div_any or div_e
        alpha_sum += a_e
        n_alpha += n_e
        if val_e and math.isfinite(lw_e):
            if rng.random() < math.exp(min(0.0, lw_e - lw_old)):
                q_curr, u_curr, g_curr, h_curr = q_e, u_e, g_e, h_e
        lw_old = float(np.logaddexp(lw_old, lw_e))
        rho = rho + rho_e
        depth += 1
        if not ut_ok:
            break

    return q_curr, u_curr, g_curr, alpha_sum / max(n_alpha, 1), depth, div_any, n_alpha, h_curr


def run_nuts_chain(
    fn, q_init: np.ndarray, n_warmup: int, n_draws: int, rng,
    target_accept: float = 0.85, max_depth: int = 10,
    adapt_eps: bool = True, adapt_mass: bool = True, fixed_eps: float | None = None,
) -> dict:
    """Jalankan satu rantai NUTS lengkap dengan adaptasi dual averaging & massa diagonal berjendela."""
    D = q_init.shape[0]
    inv_m = np.ones(D, dtype=float)
    sqrt_m = np.ones(D, dtype=float)
    q = q_init.copy()
    u, grad = fn(q)
    eps = fixed_eps if fixed_eps is not None else find_reasonable_epsilon(q, u, grad, inv_m, fn, rng, 0.1)

    init_buf, slow_len, win_ends = build_stan_warmup_windows(n_warmup, adapt_mass and adapt_eps)
    mu_da = math.log(10.0 * eps)
    log_eps_bar = math.log(eps)
    h_bar = 0.0
    gamma_da, t0_da, kappa_da = 0.05, 10.0, 0.75
    m_da = 0
    win_samples: list[np.ndarray] = []

    draws = np.empty((n_draws, D), dtype=float)
    energies = np.empty(n_draws, dtype=float)
    depths = np.empty(n_draws, dtype=int)
    divs = np.zeros(n_draws, dtype=bool)
    n_leapfrogs = np.empty(n_draws, dtype=int)
    eps_trace = np.empty(max(1, n_warmup), dtype=float)

    for it in range(n_warmup + n_draws):
        q, u, grad, alpha, d, div, n_lf, h_curr = nuts_step(
            q, u, grad, eps, inv_m, sqrt_m, fn, rng, max_depth=max_depth
        )
        if it < n_warmup:
            eps_trace[it] = eps
            if adapt_eps:
                m_da += 1
                eta_da = 1.0 / (m_da + t0_da)
                h_bar = (1.0 - eta_da) * h_bar + eta_da * (target_accept - alpha)
                log_eps = mu_da - (math.sqrt(m_da) / gamma_da) * h_bar
                eps = float(np.clip(math.exp(log_eps), 1e-5, 5.0))
                w_da = m_da ** (-kappa_da)
                log_eps_bar = w_da * log_eps + (1.0 - w_da) * log_eps_bar

                if adapt_mass and init_buf <= it < init_buf + slow_len:
                    win_samples.append(q.copy())
                    if it in win_ends and len(win_samples) >= 5:
                        arr = np.asarray(win_samples)
                        nw = arr.shape[0]
                        var_q = np.var(arr, axis=0, ddof=1)
                        inv_m = np.clip((nw / (nw + 5.0)) * var_q + (5.0 / (nw + 5.0)) * 1e-3, 1e-5, 1e4)
                        sqrt_m = 1.0 / np.sqrt(inv_m)
                        win_samples.clear()
                        eps = find_reasonable_epsilon(q, u, grad, inv_m, fn, rng, math.exp(log_eps_bar))
                        mu_da = math.log(5.0 * eps)
                        log_eps_bar = math.log(eps)
                        h_bar = 0.0
                        m_da = 0
                if it == n_warmup - 1:
                    eps = float(np.clip(math.exp(log_eps_bar), 1e-5, 5.0))
        else:
            idx = it - n_warmup
            draws[idx] = q
            energies[idx] = h_curr
            depths[idx] = d
            divs[idx] = div
            n_leapfrogs[idx] = n_lf

    return {
        "draws": draws,
        "energies": energies,
        "depths": depths,
        "divs": divs,
        "n_leapfrogs": n_leapfrogs,
        "eps": eps,
        "inv_m": inv_m,
        "eps_trace": eps_trace,
    }


def split_rhat_and_ess(chains: np.ndarray) -> dict[str, float]:
    """Diagnostik Vehtari et al. (2021): rank-normalized split-Rhat, bulk-ESS, dan tail-ESS."""
    M, N = chains.shape
    nh = N // 2
    split = np.concatenate([chains[:, :nh], chains[:, nh:2 * nh]], axis=0)
    S, n = split.shape

    def z_norm(x: np.ndarray) -> np.ndarray:
        r = rankdata(x.ravel(), method="average").reshape(x.shape)
        return ndtri((r - 0.375) / (S * n + 0.25))

    def rhat_ess_raw(z: np.ndarray) -> tuple[float, float]:
        means = np.mean(z, axis=1)
        W = float(np.mean(np.var(z, axis=1, ddof=1)))
        B = float(n * np.var(means, ddof=1))
        var_hat = ((n - 1.0) / n) * W + B / n
        if W <= 0.0 or var_hat <= 0.0:
            return 1.0, float(S * n)
        rhat = math.sqrt(var_hat / W)
        zc = z - means[:, None]
        f = np.fft.rfft(zc, n=2 * n, axis=1)
        acov = np.fft.irfft(f * np.conj(f), n=2 * n, axis=1)[:, :n] / n
        rho_hat = 1.0 - (W - np.mean(acov, axis=0)) / var_hat
        max_pairs = n // 2
        P = rho_hat[0:2 * max_pairs:2] + rho_hat[1:2 * max_pairs:2]
        pos = np.where(P <= 0.0)[0]
        k = int(pos[0]) if pos.size > 0 else max_pairs
        P_use = np.minimum.accumulate(P[:k]) if k > 1 else P[:max(k, 1)]
        tau = max(-1.0 + 2.0 * float(np.sum(P_use)), 1.0 / math.log10(S * n))
        return rhat, float(S * n) / tau

    rhat_b, ess_b = rhat_ess_raw(z_norm(split))
    rhat_f, _ = rhat_ess_raw(z_norm(np.abs(split - np.median(split))))
    _, ess_q05 = rhat_ess_raw((split <= np.quantile(split, 0.05)).astype(float))
    _, ess_q95 = rhat_ess_raw((split <= np.quantile(split, 0.95)).astype(float))
    return {"split_rhat": max(rhat_b, rhat_f), "ess_bulk": ess_b, "ess_tail": min(ess_q05, ess_q95)}


def compute_ebfmi(energies: np.ndarray) -> float:
    """Energy Bayesian Fraction of Missing Information (Betancourt 2016) per rantai."""
    denom = float(np.sum((energies - np.mean(energies)) ** 2))
    if denom <= 0.0:
        return 1.0
    return float(np.sum(np.diff(energies) ** 2) / denom)


In [7]:
# Sel Uji & Verifikasi Bagian 4 (c3): NUTS pada Gaussian Berkorelasi 50 Dimensi
D_G50 = 50
idx_50 = np.arange(D_G50)
scales_50 = np.linspace(0.4, 2.5, D_G50)
corr_50 = 0.75 ** np.abs(idx_50[:, None] - idx_50[None, :])
COV_G50 = (scales_50[:, None] * scales_50[None, :]) * corr_50
PREC_G50 = np.linalg.inv(COV_G50)
MU_G50 = np.linspace(-2.0, 2.0, D_G50)


def gaussian_50d_neg_log_post(q: np.ndarray) -> tuple[float, np.ndarray]:
    diff = q - MU_G50
    grad = PREC_G50 @ diff
    return 0.5 * float(np.dot(diff, grad)), grad


t0_g50 = time.perf_counter()
g50_outs = []
for cs in SS_G50.spawn(4):
    rc = np.random.default_rng(cs)
    g50_outs.append(
        run_nuts_chain(gaussian_50d_neg_log_post, rc.normal(size=D_G50), n_warmup=400, n_draws=1000, rng=rc, target_accept=0.80, max_depth=10)
    )
T_G50_SEC = time.perf_counter() - t0_g50

g50_chains = np.stack([o["draws"] for o in g50_outs], axis=0)  # (4, 1000, 50)
g50_flat = g50_chains.reshape(-1, D_G50)
g50_ess = np.array([split_rhat_and_ess(g50_chains[:, :, j])["ess_bulk"] for j in range(D_G50)])
g50_rhat = np.array([split_rhat_and_ess(g50_chains[:, :, j])["split_rhat"] for j in range(D_G50)])
g50_divs = sum(int(np.sum(o["divs"])) for o in g50_outs)

g50_se = np.sqrt(np.diag(COV_G50) / g50_ess)
C3_MAX_MEAN_Z = float(np.max(np.abs(np.mean(g50_flat, axis=0) - MU_G50) / g50_se))
cov_g50_hat = np.cov(g50_flat, rowvar=False)
C3_MAX_DIAG_COV_REL = float(np.max(np.abs(np.diag(cov_g50_hat) - np.diag(COV_G50)) / np.diag(COV_G50)))
C3_MED_ESS_PER_SEC = float(np.median(g50_ess) / T_G50_SEC)
C3_MIN_ESS_PER_SEC = float(np.min(g50_ess) / T_G50_SEC)

print(
    f"[c3] Gaussian 50-D ({T_G50_SEC:.2f}s, div={g50_divs}): "
    f"maks |Z_mean| = {C3_MAX_MEAN_Z:.3f} (<= 4.0) | "
    f"maks galat rel var diag = {C3_MAX_DIAG_COV_REL:.4f} (<= 0.15) | "
    f"maks Rhat = {np.max(g50_rhat):.4f} | ESS/s median = {C3_MED_ESS_PER_SEC:.1f} (min = {C3_MIN_ESS_PER_SEC:.1f})"
)
assert C3_MAX_MEAN_Z <= 4.0 and C3_MAX_DIAG_COV_REL <= 0.15 and g50_divs == 0, "Pelanggaran c3!"


[c3] Gaussian 50-D (1.83s, div=0): maks |Z_mean| = 2.876 (<= 4.0) | maks galat rel var diag = 0.0642 (<= 0.15) | maks Rhat = 1.0027 | ESS/s median = 1782.4 (min = 1656.9)
